# 13 — Final Test Dataset Model Comparison

**Objective:** Evaluate the five previously trained pork-packaging inspection configurations on the **same untouched test images**, reporting image-level PASS/FAIL performance.

The four trained architectures and their evaluation rules are taken from the training and validation notebooks:

| Configuration | Saved model | Image-level rule |
|---|---|---|
| Whole-image Logistic Regression | `baseline_logistic_regression.joblib` | Standard predicted class |
| Regional Logistic Regression | `defect_region_classifier.joblib` | FAIL if **more than 37** of the 32×32 patches are defective (Notebook 07) |
| Whole-image Neural Network | `baseline_neural_network_identity.joblib` | Standard predicted class (Notebook 09) |
| Regional Neural Network | `binary_region_nn.joblib` | FAIL if defective patch count exceeds the saved threshold (Notebook 11) |
| Identity NN (PASS threshold 0.45) | `baseline_neural_network_identity.joblib` | PASS if `P(PASS) >= 0.45` (Notebook 10) |

**Test-set policy:** Do not fit models, change hyperparameters, select new decision thresholds, or recalculate the train/validation split using the test results. The annotation JSON defines evaluation labels only; features and predictions must not use defect annotations.

**Controlled threshold comparison:** Both whole-image neural-network configurations use the same saved identity-activation pipeline from Notebook 08. Only their decision rule differs. The 0.45 PASS threshold is a comparison setting; validate and justify the chosen threshold in Notebook 10 before final testing. The configuration is not labelled an improvement until supported by validation evidence.


## Imports, project paths and fixed evaluation settings

Run this notebook from your existing `notebooks/` directory, as with Notebooks 00–13. The notebook uses model files already saved by those notebooks.

In [ ]:
# Core Python utilities.
from pathlib import Path
import json
import time

# Numerical processing, image loading, plotting and model deserialisation.
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import joblib

# Evaluation utilities (used consistently for every model).
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Display tables in Jupyter without relying on implicit notebook globals.
from IPython.display import display

# Notebook location: <project_root>/notebooks/13_final_test_model_comparison.ipynb
PROJECT_ROOT = Path('..')
TEST_DIR = PROJECT_ROOT / 'data' / 'cleaned' / 'test'
TEST_COCO_PATH = TEST_DIR / '_annotations.coco.json'
MODEL_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'

# Exactly the saved artefacts used in Notebooks 04, 07, 09, 11 and 12.
MODEL_PATHS = {
    'Whole-image Logistic Regression': MODEL_DIR / 'baseline_logistic_regression.joblib',
    'Regional Logistic Regression': MODEL_DIR / 'defect_region_classifier.joblib',
    'Whole-image Neural Network': MODEL_DIR / 'baseline_neural_network_identity.joblib',
    'Regional Neural Network': MODEL_DIR / 'binary_region_nn.joblib',
    'Identity NN (PASS threshold 0.45)': MODEL_DIR / 'baseline_neural_network_identity.joblib',
}

# Regional neural network: read the defect count from Notebook 11's JSON file.
REGIONAL_NN_CONFIG = MODEL_DIR / 'binary_region_nn_threshold.json'

# Regional logistic regression: constant used in Notebook 07.
REGIONAL_LR_MAX_DEFECTS = 37

# Identity model: comparison threshold; confirm on validation in Notebook 10 before final testing.
IDENTITY_NN_PASS_THRESHOLD = 0.45

# Both regional models use the 32×32 grid established in Notebook 05.
PATCH_SIZE = 32

WHOLE_FEATURE_COLUMNS = [
    'mean_red', 'mean_green', 'mean_blue',
    'std_red', 'std_green', 'std_blue'
]
REGION_FEATURE_COLUMNS = [
    'mean_r', 'mean_g', 'mean_b',
    'std_r', 'std_g', 'std_b'
]

# The order is fixed so confusion matrices can be read consistently.
CLASS_ORDER = ['FAIL', 'PASS']

# Confirm the notebook is being run from the expected working directory.
required_files = [TEST_COCO_PATH, REGIONAL_NN_CONFIG, *MODEL_PATHS.values()]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Some required files are missing. Check your notebook working directory and model names:\n'
        + '\n'.join(missing_files)
    )

print('All test annotations and four distinct saved model files were found.')

## Load all five models and their fixed decision rules

The saved sklearn pipelines include their fitted scalers when the training notebook used standardisation. **Do not fit or re-fit any scaler here.**

In [ ]:
# Load each distinct artifact once; both identity configurations share one object.
loaded_models = {
    path: joblib.load(path) for path in dict.fromkeys(MODEL_PATHS.values())
}
models = {name: loaded_models[path] for name, path in MODEL_PATHS.items()}
identity_model = models['Whole-image Neural Network']
if identity_model.named_steps['classifier'].activation != 'identity':
    raise ValueError('Expected the identity model; regenerate it with Notebook 08.')
assert models['Identity NN (PASS threshold 0.45)'] is identity_model

# Recover the regional NN threshold selected in Notebook 11.
with open(REGIONAL_NN_CONFIG, 'r', encoding='utf-8') as f:
    regional_nn_config = json.load(f)

REGIONAL_NN_MAX_DEFECTS = int(regional_nn_config['max_defects'])
REGIONAL_NN_PATCH_SIZE = int(regional_nn_config['patch_size'])
assert REGIONAL_NN_PATCH_SIZE == PATCH_SIZE, (
    'The regional NN was trained with a different patch size.'
)

# Check that both whole-image estimators expose the expected descriptive classes.
for model_name in [
    'Whole-image Logistic Regression',
    'Whole-image Neural Network',
    'Identity NN (PASS threshold 0.45)',
]:
    assert set(models[model_name].classes_) == {'PASS', 'FAIL'}, (
        f'Unexpected class labels in {model_name}: {models[model_name].classes_}'
    )

# Regional models use 0 for normal and 1 for defective patches.
for model_name in ['Regional Logistic Regression', 'Regional Neural Network']:
    assert set(models[model_name].classes_) == {0, 1}, (
        f'Unexpected region classes in {model_name}: {models[model_name].classes_}'
    )

print('Loaded configurations:')
for name, path in MODEL_PATHS.items():
    print(f'  {name}: {path.name}')
print(f'  Regional Logistic Regression: FAIL if count > {REGIONAL_LR_MAX_DEFECTS}')
print(f'  Regional Neural Network: FAIL if count > {REGIONAL_NN_MAX_DEFECTS}')
print(f'  Identity NN (PASS threshold 0.45): PASS if probability >= {IDENTITY_NN_PASS_THRESHOLD}')

# Record the neural network activation for transparency in the report.
for name in ['Whole-image Neural Network', 'Identity NN (PASS threshold 0.45)']:
    estimator = models[name]
    mlp = estimator.named_steps['classifier'] if hasattr(estimator, 'named_steps') else estimator
    print(f'  {name} activation: {mlp.activation}; layers: {mlp.hidden_layer_sizes}')

## Read test annotations and establish the ground-truth labels

Match the rule used in Notebooks 02, 05, 10 and 13: **an image is FAIL if it has at least one COCO defect annotation, otherwise PASS**. These labels are used only to evaluate predictions—not as model inputs.

In [ ]:
# Read only the test partition's COCO metadata/annotations.
with open(TEST_COCO_PATH, 'r', encoding='utf-8') as f:
    test_coco = json.load(f)

test_images = test_coco['images']
annotated_image_ids = {
    annotation['image_id'] for annotation in test_coco['annotations']
}

# Create exactly one image-level target record per COCO image.
test_labels = pd.DataFrame([
    {
        'image_id': image['id'],
        'filename': image['file_name'],
        'actual_label': 'FAIL' if image['id'] in annotated_image_ids else 'PASS',
    }
    for image in test_images
])

# Catch missing, repeated, or incorrectly referenced test entries early.
assert len(test_labels) > 0, 'The test annotation file contains no images.'
assert test_labels['image_id'].is_unique, 'Duplicate COCO image IDs in test set.'
assert test_labels['filename'].is_unique, 'Duplicate filenames in test set.'
assert set(annotated_image_ids).issubset(set(test_labels['image_id'])), (
    'Some test annotations refer to nonexistent test image IDs.'
)

missing_images = [
    name for name in test_labels['filename']
    if not (TEST_DIR / name).is_file()
]
if missing_images:
    raise FileNotFoundError(
        f'{len(missing_images)} test images are missing; examples: {missing_images[:5]}'
    )

print(f'Test images: {len(test_labels)}')
print('Ground-truth label distribution:')
print(test_labels['actual_label'].value_counts().reindex(CLASS_ORDER, fill_value=0))
if test_labels['actual_label'].nunique() < 2:
    print('WARNING: Only one class is present. Balanced accuracy and macro F1 require extra care.')

display(test_labels.head())

## Feature-extraction functions

Reuse the six RGB mean and standard-deviation features from Notebook 02 for **whole images** and Notebook 05 for **non-overlapping 32×32 regions**. Partial edge patches are retained, exactly as in the regional validation notebooks.

We use Pillow RGB image arrays, as in the original feature-extraction notebooks. A model receives only the six feature columns it was trained on.

In [ ]:
# Make the project package importable when running from notebooks/.
import sys
from pathlib import Path

_SRC_PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "src" / "__init__.py").is_file()
)
if str(_SRC_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_SRC_PROJECT_ROOT))

from src.feature_extraction import extract_whole_image_rgb_features


from src.patch_processing import iter_rgb_patches as extract_patches


from src.feature_extraction import extract_region_rgb_features


## Extract both feature types from the same test images

Each image is loaded once and contributes **one** whole-image feature row and **many** region-feature rows. Image IDs are retained so patch counts can be mapped back to the correct original image.

In [ ]:
whole_feature_records = []
region_feature_records = []

feature_start = time.perf_counter()

for index, image_info in enumerate(test_images, start=1):
    image_id = image_info['id']
    filename = image_info['file_name']

    # Match Notebook 02: open each image with Pillow and force RGB.
    with Image.open(TEST_DIR / filename) as img:
        image_rgb = np.asarray(img.convert('RGB'))

    # Generate one whole-image feature vector (six predictors).
    whole_feature_records.append({
        'image_id': image_id,
        'filename': filename,
        **extract_whole_image_rgb_features(image_rgb),
    })

    # Generate one regional feature vector for every non-overlapping patch.
    # Only pixels are used here: no COCO bounding boxes or actual labels.
    for patch_number, patch_rgb in enumerate(extract_patches(image_rgb, PATCH_SIZE)):
        region_feature_records.append({
            'image_id': image_id,
            'patch_number': patch_number,
            **extract_region_rgb_features(patch_rgb),
        })

    # Lightweight progress indicator for larger test partitions.
    if index % 25 == 0 or index == len(test_images):
        print(f'Features extracted: {index}/{len(test_images)} images')

whole_features_df = pd.DataFrame(whole_feature_records)
region_features_df = pd.DataFrame(region_feature_records)
feature_extraction_seconds = time.perf_counter() - feature_start

# Protect against missing data and unintentional omissions.
assert whole_features_df['image_id'].is_unique
assert whole_features_df['image_id'].tolist() == test_labels['image_id'].tolist()
assert region_features_df['image_id'].nunique() == len(test_images)
assert not whole_features_df[WHOLE_FEATURE_COLUMNS].isna().any().any()
assert not region_features_df[REGION_FEATURE_COLUMNS].isna().any().any()

X_whole_test = whole_features_df[WHOLE_FEATURE_COLUMNS]
X_region_test = region_features_df[REGION_FEATURE_COLUMNS]

print(f'Whole-image feature rows: {len(X_whole_test):,}')
print(f'Region feature rows: {len(X_region_test):,}')
print(f'Feature extraction (shared): {feature_extraction_seconds:.2f} s')
display(whole_features_df.head())

## Generate predictions for the three whole-image configurations

The logistic regression and identity neural-network baseline use their standard `predict()` methods. The threshold comparison reuses the exact same fitted identity estimator with `predict_proba()` and a PASS threshold of 0.45. No weights, scaler or input features are changed. Confirm the threshold using Notebook 10's validation experiment before final test evaluation.


In [ ]:
# Assemble predictions into one row per test image, starting with COCO targets.
predictions_df = test_labels.copy()

# Track model inference time separately from shared RGB feature extraction.
inference_seconds = {}

# 1. Baseline logistic regression (Notebook 04).
name = 'Whole-image Logistic Regression'
start = time.perf_counter()
predictions_df[name] = models[name].predict(X_whole_test)
inference_seconds[name] = time.perf_counter() - start

# 3. Whole-image neural network (Notebook 09).
name = 'Whole-image Neural Network'
start = time.perf_counter()
predictions_df[name] = models[name].predict(X_whole_test)
inference_seconds[name] = time.perf_counter() - start

# 5. Optimised neural network (Notebook 10).
name = 'Identity NN (PASS threshold 0.45)'
start = time.perf_counter()
threshold_model = models[name]
pass_class_index = list(threshold_model.classes_).index('PASS')
pass_probabilities = threshold_model.predict_proba(X_whole_test)[:, pass_class_index]
predictions_df[name] = np.where(
    pass_probabilities >= IDENTITY_NN_PASS_THRESHOLD,
    'PASS',
    'FAIL',
)
inference_seconds[name] = time.perf_counter() - start

# Preserve predicted probabilities for investigating individual errors later.
predictions_df['Identity NN P(PASS)'] = pass_probabilities

print('Whole-image predictions complete.')
display(predictions_df.head())

## Generate predictions for the two regional configurations

Each regional model predicts whether a 32×32 patch is defective (`1`) or normal (`0`). We count its defective predictions by image, then apply the **fixed** allowable-defect threshold. Importantly, both comparisons use the `>` operator from the earlier notebooks.

In [ ]:
from src.model_evaluation import regional_predictions_to_image_labels


# 2. Regional logistic regression: use MAX_DEFECTS = 37 from Notebook 07.
name = 'Regional Logistic Regression'
labels, counts, seconds = regional_predictions_to_image_labels(
    models[name], REGIONAL_LR_MAX_DEFECTS, name,
    X_region_test, region_features_df, predictions_df
)
predictions_df[name] = labels
predictions_df['Regional LR defect count'] = counts
inference_seconds[name] = seconds

# 4. Regional neural network: use the saved JSON threshold from Notebook 11.
name = 'Regional Neural Network'
labels, counts, seconds = regional_predictions_to_image_labels(
    models[name], REGIONAL_NN_MAX_DEFECTS, name,
    X_region_test, region_features_df, predictions_df
)
predictions_df[name] = labels
predictions_df['Regional NN defect count'] = counts
inference_seconds[name] = seconds

# Verify all five models returned a legitimate classification for every image.
for model_name in MODEL_PATHS:
    assert not predictions_df[model_name].isna().any()
    assert set(predictions_df[model_name].unique()).issubset({'PASS', 'FAIL'})

print('All five models have been evaluated on the same test images.')
display(predictions_df.head())

## Calculate comparable image-level performance metrics

The positive inspection outcome of concern is **FAIL**. Here:
- **False PASS:** actually FAIL, predicted PASS (defective package incorrectly accepted).
- **False FAIL:** actually PASS, predicted FAIL (acceptable package rejected).

All five models are scored using the same test-image labels and the same definitions. Macro F1 gives equal weight to the PASS and FAIL classes, independently of their relative frequencies.

In [ ]:
y_true = predictions_df['actual_label']
metrics_records = []
confusion_matrices = {}

for model_name in MODEL_PATHS:
    y_pred = predictions_df[model_name]

    # Rows are actual classes; columns are predicted classes.
    # With labels=['FAIL', 'PASS'] the first row is [true FAIL, false PASS].
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)
    confusion_matrices[model_name] = cm

    # Extract the error types relevant to food-package inspection.
    false_pass = int(cm[0, 1])   # Actual FAIL incorrectly classified PASS.
    false_fail = int(cm[1, 0])   # Actual PASS incorrectly classified FAIL.

    metrics_records.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Balanced accuracy': balanced_accuracy_score(y_true, y_pred),
        'Macro F1': f1_score(
            y_true, y_pred, labels=CLASS_ORDER,
            average='macro', zero_division=0
        ),
        'FAIL recall': recall_score(
            y_true, y_pred, pos_label='FAIL', zero_division=0
        ),
        'PASS recall': recall_score(
            y_true, y_pred, pos_label='PASS', zero_division=0
        ),
        'False PASS': false_pass,
        'False FAIL': false_fail,
        # Batched model inference only; excludes shared feature extraction.
        'Inference seconds': inference_seconds[model_name],
        'Inference ms / test image': 1000 * inference_seconds[model_name] / len(test_images),
    })

comparison_df = pd.DataFrame(metrics_records).set_index('Model')

# Display a readable, ranked performance table without changing any model.
display(
    comparison_df.sort_values('Macro F1', ascending=False).style.format({
        'Accuracy': '{:.2%}',
        'Balanced accuracy': '{:.2%}',
        'Macro F1': '{:.3f}',
        'FAIL recall': '{:.2%}',
        'PASS recall': '{:.2%}',
        'Inference seconds': '{:.3f}',
        'Inference ms / test image': '{:.3f}',
    })
)

# Include the detailed class-level report for each model.
for model_name in MODEL_PATHS:
    print(f'\n--- {model_name} ---')
    print(classification_report(
        y_true, predictions_df[model_name],
        labels=CLASS_ORDER, zero_division=0, digits=3
    ))

## Display all five confusion matrices

The colour-grid layout makes false PASS and false FAIL counts directly comparable. The class order is deliberately identical in every panel.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, model_name in zip(axes.flat, MODEL_PATHS):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrices[model_name],
        display_labels=CLASS_ORDER
    )
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(model_name, fontsize=11)

# Five models occupy five of the six panels.
for ax in list(axes.flat)[len(MODEL_PATHS):]:
    ax.axis('off')

fig.suptitle('Held-out test set — image-level confusion matrices', fontsize=15)
plt.tight_layout()
plt.show()

## Plot performance side-by-side

Show class-sensitive metrics alongside false PASS counts. Accuracy alone can obscure poor recognition of the minority PASS class.

In [ ]:
# Plot four main metrics for a direct comparison.
metric_columns = ['Accuracy', 'Balanced accuracy', 'Macro F1', 'FAIL recall', 'PASS recall']
plot_df = comparison_df[metric_columns].sort_values('Macro F1', ascending=False)

ax = plot_df.plot(kind='bar', figsize=(14, 6), width=0.85)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score (0 to 1)')
ax.set_xlabel('Model')
ax.set_title('Test set — comparison of image-level classification metrics')
ax.legend(loc='lower left', bbox_to_anchor=(1.0, 0))
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

# Show absolute error counts because false PASS may be particularly costly.
ax = comparison_df[['False PASS', 'False FAIL']].plot(
    kind='bar', figsize=(12, 5), width=0.7
)
ax.set_title('Test set — incorrect acceptance and rejection counts')
ax.set_xlabel('Model')
ax.set_ylabel('Number of images')
ax.legend(title='Error type')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## Identify disagreement and inspect incorrect predictions

Compare predictions for individual packages, especially **false PASS** cases. Do not use these inspection results to revise the evaluated models; use them to explain model limitations in your final report.

In [ ]:
# Mark whether each model classified an image incorrectly.
error_df = predictions_df[['image_id', 'filename', 'actual_label']].copy()

for model_name in MODEL_PATHS:
    error_df[f'{model_name} incorrect'] = (
        predictions_df[model_name] != predictions_df['actual_label']
    )

# Count how many of the five models made an error on each image.
error_flags = [f'{name} incorrect' for name in MODEL_PATHS]
error_df['models_incorrect'] = error_df[error_flags].sum(axis=1)

# Show difficult images and their five predictions together.
most_difficult = error_df.sort_values('models_incorrect', ascending=False)
print('Number of images misclassified by at least one model:',
      int((error_df['models_incorrect'] > 0).sum()))
display(most_difficult.head(15))

# For later manual inspection, list defective images accepted by each model.
for model_name in MODEL_PATHS:
    false_pass_images = predictions_df.loc[
        (predictions_df['actual_label'] == 'FAIL') &
        (predictions_df[model_name] == 'PASS'),
        ['filename', 'actual_label', model_name]
    ]
    print(f'{model_name}: {len(false_pass_images)} false PASS images')

## Optional: display example false PASS images

Change `INSPECT_MODEL` to examine any of the five configurations. Image display is for qualitative error analysis; it does not modify the test labels or model decisions.

In [ ]:
# Choose one already-evaluated configuration for visual error inspection.
INSPECT_MODEL = 'Regional Neural Network'
MAX_EXAMPLES = 6
assert INSPECT_MODEL in MODEL_PATHS

false_pass_examples = predictions_df.loc[
    (predictions_df['actual_label'] == 'FAIL') &
    (predictions_df[INSPECT_MODEL] == 'PASS')
].head(MAX_EXAMPLES)

if false_pass_examples.empty:
    print(f'No false PASS cases for {INSPECT_MODEL}.')
else:
    n_examples = len(false_pass_examples)
    fig, axes = plt.subplots(1, n_examples, figsize=(3.5 * n_examples, 3.5))
    axes = np.atleast_1d(axes)

    for ax, (_, record) in zip(axes, false_pass_examples.iterrows()):
        with Image.open(TEST_DIR / record['filename']) as img:
            ax.imshow(img.convert('RGB'))
        ax.set_title(f"Actually FAIL, predicted PASS\n{record['filename'][:34]}...", fontsize=9)
        ax.axis('off')

    fig.suptitle(f'False PASS examples — {INSPECT_MODEL}', fontsize=13)
    plt.tight_layout()
    plt.show()

## Export final evaluation tables

Save only predictions and evaluation summaries. The original test dataset, saved models, train/validation assignments and chosen thresholds remain unchanged.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_csv = OUTPUT_DIR / 'final_test_model_comparison.csv'
prediction_csv = OUTPUT_DIR / 'final_test_image_predictions.csv'
errors_csv = OUTPUT_DIR / 'final_test_error_audit.csv'

# Retain full floating-point precision in the underlying exports.
comparison_df.reset_index().to_csv(summary_csv, index=False)
predictions_df.to_csv(prediction_csv, index=False)
error_df.to_csv(errors_csv, index=False)

print('Saved final test results:')
print(f'  {summary_csv}')
print(f'  {prediction_csv}')
print(f'  {errors_csv}')

## Conclusions and evaluation limitations

After running the notebook, write a brief results interpretation supported by the actual scores:

1. **Overall:** Which configuration has the highest test macro F1 and balanced accuracy? How does that compare with plain accuracy?
2. **Quality-control perspective:** Which model produces the fewest **false PASS** cases? What trade-off does it make in false FAIL cases?
3. **Regional vs whole-image:** Does explicitly classifying 32×32 regions help at the **package** level? Note that regional patch counts can also depend on image size.
4. **Decision thresholds:** Both neural-network configurations use the same fitted identity model. Attribute differences to the decision rule, and report how its threshold was justified on validation data before testing.
5. **Threshold-selection caveat:** Notebook 11 selected its regional NN defect-count limit from *training data*, which can be optimistic. The present held-out evaluation provides an independent check; it must not be used to select a replacement threshold.
6. **Label assumption:** PASS means **no COCO annotation**, not proof that the package is defect-free. This interpretation was established as a provisional assumption during dataset exploration.
7. **Processing time caveat:** The reported inference timings are **batched estimator prediction timings** and exclude image reading and feature extraction; they are not end-to-end deployment latencies.

**Do not use the test set repeatedly to choose or tune a model.** Use it as a final performance audit and document all decisions already made using training or validation results.
